# 05 · Time travel and rollback

**Time:** ~20 min · **You will learn:** query old versions of a table, and undo a mistake.

Every snapshot is a complete, consistent version of the table, so reading history is just reading an older snapshot.

In [ ]:
from helpers import get_spark, reset_orders

spark = get_spark("05-time-travel")
reset_orders(spark, "orders", n=6, commits=3)

snaps = spark.sql("SELECT snapshot_id, committed_at FROM demo.course.orders.snapshots ORDER BY committed_at").collect()
first, second, third = snaps
for s in snaps:
    print(s.snapshot_id, s.committed_at)

## 1. Read an old snapshot by id

In [ ]:
spark.sql(f"SELECT COUNT(*) AS rows_then FROM demo.course.orders VERSION AS OF {first.snapshot_id}").show()
spark.sql("SELECT COUNT(*) AS rows_now FROM demo.course.orders").show()

## 2. Read as of a timestamp

In [ ]:
ts = second.committed_at.strftime("%Y-%m-%d %H:%M:%S.%f")
spark.sql(f"SELECT COUNT(*) AS rows_at_ts FROM demo.course.orders TIMESTAMP AS OF '{ts}'").show()

## 3. The DataFrame API equivalent

In [ ]:
old = spark.read.option("snapshot-id", first.snapshot_id).table("demo.course.orders")
print("rows in first snapshot:", old.count())

## 4. Oops: undo a mistake
Someone runs a `DELETE` without a `WHERE`. Because the old snapshot still exists, we can **roll back**.

In [ ]:
good_snapshot = spark.sql("SELECT snapshot_id FROM demo.course.orders.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0][0]
spark.sql("DELETE FROM demo.course.orders")
print("rows after the accident:", spark.table("demo.course.orders").count())

In [ ]:
spark.sql(f"CALL demo.system.rollback_to_snapshot('course.orders', {good_snapshot})").show()
print("rows after rollback:", spark.table("demo.course.orders").count())

`rollback_to_snapshot` only moves the table's *current snapshot pointer*; no data is copied.
The bad `DELETE` snapshot still exists in history (check `.snapshots`), it is simply not current any more.

### Your turn

Run `UPDATE demo.course.orders SET unit_price = 0`. Then look at the `.history` table, find the snapshot
that was current **before** the update, and use `CALL demo.system.set_current_snapshot(...)` to go back.
Assert that no row has `unit_price = 0` afterwards.

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
spark.sql("UPDATE demo.course.orders SET unit_price = 0")
hist = spark.sql("SELECT snapshot_id FROM demo.course.orders.history ORDER BY made_current_at").collect()
previous = hist[-2].snapshot_id
spark.sql(f"CALL demo.system.set_current_snapshot('course.orders', {previous})")
assert spark.table("demo.course.orders").filter("unit_price = 0").count() == 0
```

</details>

### Recap
* `VERSION AS OF <snapshot id>` and `TIMESTAMP AS OF '<time>'` query history; `option("snapshot-id", ...)` does the same in PySpark.
* `rollback_to_snapshot` / `set_current_snapshot` undo mistakes instantly.
* History is kept until you **expire** snapshots (lesson 09), so plan retention deliberately.

Next: **06 · Schema and partition evolution**.